# Spikes2Slope | Meeting walkthrough with Liz

**November 3, 2026 · Allen Visual Coding Neuropixels · Q1: Can extracellular electrophysiological features distinguish optotagging-supported Pvalb, Sst, and Vip populations better than chance?**

**How to use this notebook:** Run cells in order, pause at each **Ask Liz** checkpoint, and record decisions at the end. The goal is a clear scientific conversation, **not** to introduce a new experimental protocol during the meeting.

**One-minute overview to explain aloud**

> We used the Allen Institute's published optotagging tutorial criterion to identify putatively light-responsive units in Pvalb-, Sst-, and Vip-Cre sessions. We matched those unit IDs to Allen's spike-sorted unit metadata, extracted Allen-computed waveform features, and trained exploratory SVM models using held-out sessions. The models show modest discrimination, but the experimental label definition and an appropriate chance-level test still need agreement. I'd like to confirm the label definition and the next milestone before extending identity predictions to untagged units.

**Central distinction:** Cre line identifies the experimental targeting context; a rate-threshold-positive unit is an **optotagging candidate**, not automatically a molecularly confirmed neuron. All proposed model labels are conditional on accepting that candidate definition.

**Data sources:** [Allen optotagging tutorial](https://allensdk.readthedocs.io/en/latest/_static/examples/nb/ecephys_optotagging.html) · [Allen Visual Coding documentation](https://allensdk.readthedocs.io/en/latest/visual_coding_neuropixels.html) · project notebooks `optotagcheck.ipynb`, `kilosortvalidation.ipynb`, `waveform.ipynb`, `svm.ipynb`.

**Runtime:** Reuses CSVs already produced under `/Volumes/WD_BLACK/Spikes2Slope/opto_results`; the required sections do **not** load/download NWB sessions. Rerunning the SVM is optional and uses locally saved features.


## 0. Load the existing results

**Say:** "I'm not recomputing the entire dataset today. These saved tables are the output of the original optotagging, metadata, and waveform pipelines."

**Why the tables differ:** `all_opto_unit_results.csv` contains units examined in selected visual regions of 28 optotagging sessions; `master_waveform_features.csv` contains the broader Allen QC unit table, with other units marked **unknown**—not excitatory.

**Ask Liz:** "Is this the right scoped population and set of regions for the initial Q1 question? We included VISp, VISam, VISrl, VISal, VISl, VISpm, and LGd."

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path("/Volumes/WD_BLACK/Spikes2Slope")
RESULTS = ROOT / "opto_results"
opto_path = RESULTS / "all_opto_unit_results.csv"
master_path = RESULTS / "master_waveform_features.csv"

for path in (opto_path, master_path):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing saved output: {path}\n"
            "Mount WD_BLACK or adjust ROOT to the existing cache; "
            "do not re-download raw recordings for this meeting."
        )

opto = pd.read_csv(opto_path)
master = pd.read_csv(master_path)
for col in ["candidate_tagged", "allen_qc", "snr_qc", "matched_candidate"]:
    opto[col] = opto[col].astype(bool)
master["is_opto_candidate"] = master["is_opto_candidate"].astype(bool)

assert opto["unit_id"].is_unique
assert master["unit_id"].is_unique
print(f"Analyzed units: {len(opto):,} in {opto.ecephys_session_id.nunique()} sessions")
print(f"QC-matched optotagging candidates: {opto.matched_candidate.sum():,}")
print(f"Allen QC units in master waveform table: {len(master):,}")
display(opto[["unit_id", "ecephys_session_id", "cre_line", "region",
              "n_pulses", "baseline_rate_hz", "evoked_rate_hz",
              "response_ratio", "matched_candidate"]].head(4))

## 1. How did we identify optotagging candidates?

**Say:** "The Allen SDK tutorial searches for units with more firing during a short interval after the light pulse than immediately before it. Our implementation follows that exact response-ratio recipe; we additionally restrict to acceptable spike-sorting quality."

- **Optical protocol:** `stimulus_name == "pulse"`, with duration strictly between 9 and 20 ms (the nominal 10 ms pulses).
- **Baseline:** `[-10, -2)` ms relative to optical onset, duration 8 ms.
- **Evoked:** `[+1, +9)` ms relative to optical onset, duration 8 ms.
- **Rates:** total counts across selected pulses, divided by total exposure.
- **Allen tutorial-style detector:** `evoked_rate_hz / (baseline_rate_hz + 1) > 2`. The `+1` is a **1 Hz stabilizer** in the denominator; the ratio is not a literal evoked/baseline fold change.
- **Our extra quality criterion:** Allen-style `presence_ratio >= .95`, `isi_violations <= .5`, `amplitude_cutoff <= .1`, plus `snr >= 1`. We call the intersection `matched_candidate`.

**Important:** Allen's algorithm detects candidate optical responses; Kilosort2 performs sorting and does **not** independently provide Pvalb/Sst/Vip molecular subtype labels.

**Ask Liz — most important decision today:** "Should Allen's tutorial threshold plus our QC define the reference population for a preliminary classifier, or do you want a different optotagging reliability definition before proceeding?"

In [ ]:
qc_checks = pd.DataFrame([
    ["Allen-style response threshold", int(opto["candidate_tagged"].sum())],
    ["Pass Allen quality metrics", int(opto["allen_qc"].sum())],
    ["Pass Allen QC + SNR >= 1", int(opto["snr_qc"].sum())],
    ["Response threshold AND combined QC", int(opto["matched_candidate"].sum())],
], columns=["Criterion", "Number of analyzed units"])
display(qc_checks)

# Verify the detector against values saved by optotagcheck.ipynb.
calculated_ratio = opto["evoked_rate_hz"] / (opto["baseline_rate_hz"] + 1)
assert np.allclose(opto["response_ratio"], calculated_ratio)
assert (opto["candidate_tagged"] == (calculated_ratio > 2)).all()
assert (opto["matched_candidate"] ==
        (opto["candidate_tagged"] & opto["snr_qc"])).all()
print("Stored labels match the original detector and QC logic.")

## 2. Why we should discuss the one-spike candidates

**Say:** "I found that the tutorial threshold is useful for identifying candidates, but it can be passed by a neuron that fires only once in the selected response windows. I want to check what evidence Liz considers sufficient for a reference label."

**Example:** With 45 pulses and an 8 ms response window, one spike corresponds to `1 / (45 × 0.008) = 2.78 Hz`. If the baseline count is zero, the tutorial-style ratio is **2.78**, which passes `>2`.

This is **not evidence that these units are definitively false positives**; it is evidence that the selection rule does not itself require reproducible trial-by-trial activation. Low spontaneous firing can also make statistical confirmation difficult.

**Ask Liz:** "Should we keep these as tutorial-defined candidates with a caveat, or distinguish high/low optotagging evidence for sensitivity analysis?"

In [ ]:
candidates = opto.loc[opto["matched_candidate"]].copy()
WINDOW_S = 0.008
for prefix in ("evoked", "baseline"):
    estimated = (candidates[f"{prefix}_rate_hz"]
                 * candidates["n_pulses"] * WINDOW_S)
    assert np.allclose(estimated, np.rint(estimated), atol=1e-6), (
        f"{prefix} count reconstruction failed; inspect rate definition"
    )
    candidates[f"{prefix}_spikes"] = np.rint(estimated).astype(int)

candidates["one_spike_zero_baseline"] = (
    (candidates["evoked_spikes"] == 1)
    & (candidates["baseline_spikes"] == 0)
)
candidates["two_or_fewer_evoked"] = candidates["evoked_spikes"] <= 2

reliability_summary = candidates.groupby("cre_line").agg(
    candidates=("unit_id", "size"),
    one_spike_zero_baseline=("one_spike_zero_baseline", "sum"),
    two_or_fewer_evoked=("two_or_fewer_evoked", "sum"),
    median_evoked_spikes=("evoked_spikes", "median")
)
reliability_summary["pct_two_or_fewer"] = (
    100 * reliability_summary["two_or_fewer_evoked"]
    / reliability_summary["candidates"]
)
display(reliability_summary.round(1))
print("Total one-spike/zero-baseline:",
      int(candidates["one_spike_zero_baseline"].sum()),
      "/", len(candidates))

## 3. If we provisionally accept those labels, how much data do we have?

**Say:** "We have three genetically targeted session groups. The model treats candidate units from each Cre line as one of three labels. This is a working classification target—not proof that the other, untagged units are excitatory."

**Caution:** The unit counts are not independent biological replicates. Each Cre line comes from different recording sessions and animals, which means genotype can be confounded with session, acquisition date, and hardware.

**Ask Liz:** "Is the goal specifically three-way Pvalb/Sst/Vip classification, or broader inhibitory-versus-other classification? Should we focus on VISp rather than pooling visual regions and LGd?"

In [ ]:
counts = (candidates.groupby("cre_line")
          .agg(n_candidates=("unit_id", "nunique"),
               n_sessions=("ecephys_session_id", "nunique"))
          .reindex(["Pvalb", "Sst", "Vip"]))
display(counts)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(counts.index, counts["n_candidates"])
ax.set(ylabel="QC-matched optotagging candidates",
       title="Candidate counts by Cre line (not cell census)")
for i, val in enumerate(counts["n_candidates"]):
    ax.text(i, val + 6, str(val), ha="center")
ax.set_ylim(0, counts["n_candidates"].max() * 1.17)
plt.tight_layout()
plt.show()

## 4. How are optotagged candidates linked to waveform measurements?

**Say:** "Allen performed spike sorting (Kilosort2). Each sorted unit has a unit ID and associated quality and waveform features. We joined optotagging response evidence to the same unit IDs. That is the validated correspondence."

**What this validates:** unit ID, session/region alignment, and matching sorting-quality metadata. **What this does not validate:** agreement with independent molecular labels from Kilosort; those labels are not a standard Kilosort2 output.

The waveform table uses Allen-computed features. The existing exploratory polarity check examined 30 candidate waveforms; it does **not** replace a systematic recomputation of waveform duration/half-width from raw mean waveforms. Liz's notes explicitly mention a ~20-unit manual check.

**Ask Liz:** "Is the unit-ID/metadata correspondence enough for the Kilosort check you intended, or did you mean comparing waveform-based fast/regular-spiking categories? Would you like the ~20-unit duration and half-width recalculation before moving ahead?"

In [ ]:
check = candidates[["unit_id", "ecephys_session_id", "region"]].merge(
    master[["unit_id", "ecephys_session_id",
            "ecephys_structure_acronym", "cell_type_label"]],
    on="unit_id", how="left", validate="one_to_one",
    suffixes=("_opto", "_waveform"), indicator=True
)
audit = pd.Series({
    "Missing candidate unit IDs in master": int((check["_merge"] != "both").sum()),
    "Session mismatches": int((check["ecephys_session_id_opto"]
                               != check["ecephys_session_id_waveform"]).sum()),
    "Region mismatches": int((check["region"]
                              != check["ecephys_structure_acronym"]).sum()),
    "Cre-label mismatches": int((check["cre_line"]
                                 if "cre_line" in check else pd.Series([], dtype=str)).size) 
})
# The Cre label is verified by joining to optotagging source, not inferred from Kilosort.
audit = audit.drop("Cre-label mismatches")
display(audit.rename("Count").to_frame())
print("All remaining master rows labeled 'unknown' are unlabeled, not excitatory.")

## 5. What do the waveform features look like?

**Say:** "Our biological expectation is that Pvalb candidates often show narrower waveforms than Sst and Vip candidates, but distributions overlap. We tested whether that information is sufficiently discriminative, not whether waveform shape uniquely determines molecular identity."

The plots below use **Allen-computed waveform features** (waveform duration and half-width), rather than raw voltage traces. They are useful for describing waveform *shape metrics* without loading large NWB files. The original `waveform.ipynb` contains supplementary waveform polarity checks.

**Ask Liz:** "Do you want us to prioritize waveform-only features, or include spike-train statistics such as firing rate and autocorrelograms? Should we separate by cortical region/layer before classification?"

In [ ]:
labels = ["Pvalb", "Sst", "Vip"]
tagged = master.loc[
    master["is_opto_candidate"]
    & master["cell_type_label"].isin(labels)
].copy()

features = ["waveform_duration", "waveform_halfwidth",
            "waveform_PT_ratio", "waveform_repolarization_slope",
            "waveform_recovery_slope", "waveform_amplitude",
            "waveform_spread"]
assert all(col in tagged.columns for col in features)
display(
    tagged.groupby("cell_type_label")[["waveform_duration", "waveform_halfwidth"]]
    .agg(["count", "median"]).reindex(labels).round(3)
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, feature, title in zip(
    axes,
    ["waveform_duration", "waveform_halfwidth"],
    ["Duration", "Half-width"]
):
    arrays = [tagged.loc[tagged.cell_type_label == label, feature]
              .dropna().to_numpy() for label in labels]
    ax.boxplot(arrays, tick_labels=labels, showfliers=False)
    ax.set(title=title, xlabel="Cre line (candidate label)",
           ylabel=f"{feature} (ms)")
fig.suptitle("Allen-computed waveform timing metrics")
plt.tight_layout()
plt.show()

### The two-feature geometry Liz specifically requested

A duration-versus-half-width scatterplot illustrates both between-class tendencies and overlap. Subsampling is only for **plot readability**, not for training. A visible separation is descriptive and does not establish predictive performance.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for label in labels:
    subset = tagged.loc[
        tagged["cell_type_label"] == label,
        ["waveform_duration", "waveform_halfwidth"]
    ].dropna()
    subset = subset.sample(n=min(250, len(subset)), random_state=42)
    ax.scatter(subset["waveform_duration"], subset["waveform_halfwidth"],
               label=label, s=16, alpha=0.5)
ax.set(xlabel="Waveform duration (ms)",
       ylabel="Waveform half-width (ms)",
       title="Timing features overlap across candidate classes")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Preliminary SVMs: can these features predict the candidate label?

**Say:** "We used seven Allen waveform features to classify the three optotagging candidate groups. We also checked whether adding firing rate helped. We evaluated performance on held-out recording sessions rather than randomly splitting individual units."

**Important CV detail:** There is only one Cre label per session. We stratify **sessions** by Cre line into five folds, then hold out all units from test sessions. This avoids the most obvious same-session train/test leakage, but cannot disentangle genotype from other session/animal factors.

**Measures:** Balanced accuracy averages recall across the three classes, so an uninformative three-class prediction has a **nominal** reference of 1/3; this is **not** yet a permutation-derived significance threshold. Macro-F1 averages per-class precision/recall tradeoffs.

**Previously observed exploratory results:** linear SVM 0.400 balanced accuracy; RBF SVM 0.444; RBF + firing rate 0.452. We will **recalculate** these below rather than treating historical numbers as guaranteed reproducible.

**Ask Liz:** "Is a held-out-session model answering Q1 as you intended? Is the next milestone a session/animal-aware permutation test, a feature validity check, or a richer model such as PhysMAP?"

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, LinearSVC
from sklearn.base import clone
from sklearn.metrics import (
    balanced_accuracy_score, f1_score, confusion_matrix,
    ConfusionMatrixDisplay
)

model_data = tagged.reset_index(drop=True)
X = model_data[features]
y = model_data["cell_type_label"]
groups = model_data["ecephys_session_id"]

session_labels = model_data.groupby("ecephys_session_id")["cell_type_label"].first()
assert (model_data.groupby("ecephys_session_id")["cell_type_label"]
        .nunique() == 1).all()

session_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = []
for train_s, test_s in session_cv.split(
    session_labels.index, session_labels.values
):
    train_sessions = session_labels.index[train_s]
    test_sessions = session_labels.index[test_s]
    train_idx = np.flatnonzero(groups.isin(train_sessions))
    test_idx = np.flatnonzero(groups.isin(test_sessions))
    assert set(groups.iloc[train_idx]).isdisjoint(
        set(groups.iloc[test_idx])
    )
    folds.append((train_idx, test_idx))

print(f"{len(model_data):,} candidates; {groups.nunique()} sessions; "
      f"{len(folds)} session-held-out folds")
display(pd.DataFrame({
    "Cre line": labels,
    "Units": [int((y == label).sum()) for label in labels],
    "Sessions": [int((session_labels == label).sum()) for label in labels]
}))

### Model comparison: waveform-only versus waveform plus firing rate

Models follow the pipelines in `svm.ipynb`: imputation, standard scaling, class-weighted linear or RBF SVM. All preprocessing is fitted **inside each training fold**. These hyperparameters are exploratory and not nested-tuned.

In [ ]:
models = {
    "Linear — waveform": (
        make_pipeline(
            SimpleImputer(strategy="median"), StandardScaler(),
            LinearSVC(class_weight="balanced", random_state=42, max_iter=10000)
        ), features
    ),
    "RBF — waveform": (
        make_pipeline(
            SimpleImputer(strategy="median"), StandardScaler(),
            SVC(kernel="rbf", C=1.0, gamma="scale",
                class_weight="balanced")
        ), features
    ),
    "RBF — waveform + firing rate": (
        make_pipeline(
            SimpleImputer(strategy="median"), StandardScaler(),
            SVC(kernel="rbf", C=1.0, gamma="scale",
                class_weight="balanced")
        ), features + ["firing_rate"]
    )
}

model_summaries = []
predictions = {}
for name, (estimator, columns) in models.items():
    fold_scores = []
    oof = np.empty(len(model_data), dtype=object)
    for fold_i, (train_idx, test_idx) in enumerate(folds, start=1):
        fitted = clone(estimator)
        fitted.fit(model_data.iloc[train_idx][columns], y.iloc[train_idx])
        pred = fitted.predict(model_data.iloc[test_idx][columns])
        oof[test_idx] = pred
        fold_scores.append({
            "fold": fold_i,
            "balanced_accuracy": balanced_accuracy_score(
                y.iloc[test_idx], pred
            ),
            "macro_f1": f1_score(
                y.iloc[test_idx], pred, average="macro", labels=labels
            )
        })
    score_df = pd.DataFrame(fold_scores)
    predictions[name] = oof
    model_summaries.append({
        "model": name,
        "mean_balanced_accuracy": score_df["balanced_accuracy"].mean(),
        "mean_macro_f1": score_df["macro_f1"].mean()
    })
model_summary = pd.DataFrame(model_summaries)
display(model_summary.round(3))
print("Nominal 3-class balanced-accuracy reference: 0.333; "
      "NO permutation test performed here.")

### Error structure matters more than the headline score

Use row-normalized out-of-fold confusion matrix for the best *exploratory* RBF+rate model. Each row is a candidate class and sums to 1.

**Say:** "Even if accuracy is above the nominal reference, we need to inspect whether it distinguishes all three populations or mostly separates Pvalb from the overlapping Sst/Vip groups."

In [ ]:
best_name = "RBF — waveform + firing rate"
cm = confusion_matrix(
    y, predictions[best_name], labels=labels, normalize="true"
)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=labels).plot(
    ax=ax, cmap="Blues", values_format=".2f", colorbar=False
)
ax.set_title("Out-of-fold predictions — held-out sessions")
plt.tight_layout()
plt.show()
display(pd.DataFrame(
    {"class": labels, "recall": np.diag(cm)}
).round(3))

## 7. Optional methodological caveat: optical response strength differs across sessions

**Say:** "After the first SVM, we audited how much light-response evidence supported the labels. Some candidates passed with only one evoked spike, and later laser-period sessions showed stronger evoked spike counts among selected candidates."

Allen's tutorial describes a transition from LED to laser hardware; the session-ID split used in our audit is `789848216`. This is an *observational period comparison*: hardware, time, mouse, and other session factors may co-vary.

For the meeting, **do not interpret the optional FDR screen as proven biological truth**: among 8,579 analyzed units, an exploratory pooled-count binomial test with BH correction marked 213 of the 1,127 original candidates. Its assumptions, power, temporal dependence, and label-selection mechanism all limit interpretation.

**Ask Liz:** "Would you like the original Allen criterion retained as our Q1 exploratory labels, with a label-confidence sensitivity analysis later?"

In [ ]:
period = opto.copy()
period["hardware_period"] = np.where(
    period["ecephys_session_id"] >= 789848216, "Laser period", "LED period"
)
period["evoked_spikes"] = np.rint(
    period["evoked_rate_hz"] * period["n_pulses"] * WINDOW_S
).astype(int)

period_candidate = period.loc[period["matched_candidate"]].copy()
period_candidate["few_spikes"] = period_candidate["evoked_spikes"] <= 2
session_audit = (period_candidate.groupby(
    ["cre_line", "hardware_period", "ecephys_session_id"]
).agg(
    n_candidates=("unit_id", "size"),
    median_evoked=("evoked_spikes", "median"),
    fraction_few=("few_spikes", "mean")
).reset_index())
display(session_audit.groupby(["cre_line", "hardware_period"]).agg(
    sessions=("ecephys_session_id", "nunique"),
    candidates=("n_candidates", "sum"),
    median_session_evoked=("median_evoked", "median"),
    median_session_fraction_few=("fraction_few", "median")
).round(3))

## 8. Decision checkpoint with Liz

### What we can claim now
- The optotagging **candidate algorithm** follows Allen's tutorial and is reproducible from pulse epochs and sorted-unit spike times.
- The analyzed population includes **28 Cre sessions**, **8,579 examined units**, and **1,127 optotagging candidates passing the combined response/QC criteria**.
- Candidate status is joined correctly to Allen's sorted-unit metadata; Kilosort is **not** providing independent molecular class labels.
- Pvalb candidates generally have narrower waveform-duration distributions than Sst/Vip candidates; substantial overlap remains.
- Preliminary held-out-session SVM models show **modest** discrimination; statistical significance, independent animals, nuisance factors, and label confidence remain unresolved.
- Evidence for optical responsiveness varies, and a single evoked spike can pass the original tutorial criterion.

### What we should *not* claim
- 1,127 molecularly confirmed inhibitory neurons; 914 proven false-positive candidates; or 213 definitive ground-truth neurons.
- A statistically significant or independently generalizable classifier.
- That untagged `unknown` units are excitatory.
- That waveform-derived subtype counts or firing-rate ratios directly measure synaptic E/I balance.
- That any observed relationship with the LFP aperiodic exponent has already been demonstrated.

### Questions for Liz — record answers during meeting
1. **Optotagging reference:** Is the Allen tutorial candidate definition + QC acceptable for initial Q1, or should we validate a more reliable subset first?
2. **Kilosort correspondence:** Did you mean alignment of sorted units to optotagging responses, or a separate fast/regular waveform category? Kilosort2 itself does not label PV/SST/VIP.
3. **Waveform validation:** Prioritize direct duration/half-width recomputation for ~20 units and a stronger polarity audit?
4. **Population scope:** Visual cortex only (VISp?) versus all current regions, including LGd? Should depth/layer be modeled?
5. **Statistical milestone:** Session/animal-aware permutation analysis for the existing SVM before PhysMAP and label extension?
6. **Other tasks:** Is bycycle intended for LFP oscillatory cycles rather than action potential waveform morphology? Where does K-means/PhysMAP fit in the priority order?

### Proposed next milestone — **proposal, not a decision**
If Liz accepts the tutorial labels for exploratory Q1, prioritize (i) verifying waveform feature extraction and sampling structure, then (ii) a defensible session-aware baseline/permutation evaluation. Preserve original labels and explicitly document uncertainty. If Liz wants stronger experimental reference labels first, agree on validation criteria **before** changing the SVM training set.

### Meeting notes
- **Agreed label definition:** ...
- **Target region/population:** ...
- **Immediate deliverable:** ...
- **Statistical test/validation design:** ...
- **Deferred ideas:** ...
- **Owner and check-in:** ...

**Long-term Spikes2Slope link:** Better-supported interneuron identity estimates could eventually help test how class-specific spiking relates to LFP aperiodic properties. They do **not** directly establish an E/I conductance balance measure.
